# Salary Prediction - v15 NN (Red Neuronal, Producción)
## Misma disciplina de v14, pero con Red Neuronal (TensorFlow/Keras) en vez de árboles

**Por qué esta versión existe:** el requisito del proyecto pide una **red neuronal**, no modelos de
árboles (XGBoost/Random Forest). Este notebook toma todas las correcciones de la v14 (limpieza del
target, imputación correcta, feature engineering, prevención de fuga de datos) y las combina con una
red neuronal bien construida, corrigiendo los errores que tenía `Model_7` del notebook original de
8 modelos.

**Qué estaba mal en `Model_7` (causa de predicciones absurdas tipo $16):**
1. Se normalizaban las variables de entrada (`X`) pero **no** el target (`Salary`). La red tenía que
   aprender a "estirar" sus salidas desde valores cercanos a 0 hasta decenas de miles de dólares, y no
   siempre lo logra bien solo ajustando pesos.
2. `EarlyStopping` monitoreaba `loss` (el error de **entrenamiento**), no una métrica de validación
   real. Esto permite que el entrenamiento se detenga en un mínimo pobre sin detectar que el modelo no
   generaliza.
3. Nunca se limpiaron outliers/errores de captura en `Salary` (el target). Registros corruptos
   (salarios casi en $0) entrenaban al modelo a predecir casi nada.
4. No había un mecanismo para "deshacer" el escalamiento y devolver la predicción a dólares reales de
   forma explícita y verificada.

**Qué se corrige aquí:**
- Limpieza de outliers en el **target** (`Salary`) antes de entrenar (regla de negocio + IQR).
- Imputación correcta de nulos (categóricas → luego numéricas dependientes de esas categorías),
  en vez de `dropna()`.
- Se escala **tanto X como y** con `StandardScaler`, y las predicciones se revierten con
  `inverse_transform` antes de calcular métricas — así la red aprende en una escala manejable y
  nunca reportamos ni guardamos números en la escala equivocada.
- `EarlyStopping` monitorea `val_loss` (un split de **validación**, no el propio training loss),
  con `validation_split` explícito, para detectar sobreajuste de verdad.
- Se agregan las mismas features nuevas de v14 (`Age_Bin`, `Experience_Bin`, `Seniority_Score`) y se
  evita la fuga de datos de `Salary_Per_Hour`.
- Métricas completas (MAPE, MAE, RMSE, R²) en train y test, ya en dólares reales.
- Importancia de variables aproximada con **permutation importance** (los pesos de una red no son
  interpretables directamente como en un árbol).
- Guardado del modelo (`.keras`) + scalers + metadata en un `.pkl` adicional, listo para una app.

**Nota sobre el dataset:** columnas originales: `Name, Phone_Number, Date_Of_Birth, Experience, Role,
Qualification, University, Cert, Salary`. No existe columna de "Departamento"; `Role` representa más
bien el nivel de seniority (p. ej. Senior vs. otros).

In [ ]:
!pip install tensorflow pandas numpy scikit-learn matplotlib --quiet
print("Librerías instaladas")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, RegressorMixin

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)

### Configuración de GPU (opcional)

Si hay una GPU disponible, TensorFlow la detecta y usa automáticamente sin que tengamos que
configurar nada extra (a diferencia de XGBoost, Keras/TensorFlow maneja esto de forma transparente).
Esta celda solo informa qué dispositivo se va a usar, y muestra el nombre de la GPU si existe.

In [ ]:
import subprocess

def get_gpu_name():
    """Intenta obtener el nombre de la GPU vía nvidia-smi. Si no está disponible, regresa None."""
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL,
            timeout=5,
        )
        names = output.decode("utf-8").strip().split("\n")
        return names[0] if names and names[0] else None
    except Exception:
        return None

gpus = tf.config.list_physical_devices('GPU')
USE_GPU = len(gpus) > 0

if USE_GPU:
    gpu_name = get_gpu_name()
    print(f"GPU detectada por TensorFlow: {len(gpus)} dispositivo(s)")
    if gpu_name:
        print(f"Nombre de la GPU: {gpu_name}")
    else:
        print("No se pudo leer el nombre de la GPU (nvidia-smi no disponible), pero TensorFlow sí la ve.")
else:
    print("No se detectó GPU. Se entrenará en CPU (funciona igual, solo más lento).")

## 1. Carga de Datos + Limpieza de columnas irrelevantes

In [ ]:
URL = "https://raw.githubusercontent.com/oluwole-packt/datasets/main/salary_dataset.csv"
df = pd.read_csv(URL)
print("Shape original:", df.shape)

# Columnas que no aportan valor predictivo (identificadores personales)
df = df.drop(columns=['Name', 'Phone_Number'], errors='ignore')

# Crear Age a partir de la fecha de nacimiento
df['Date_Of_Birth'] = pd.to_datetime(df['Date_Of_Birth'], format='%d/%m/%Y', errors='coerce')
df['Age'] = 2025 - df['Date_Of_Birth'].dt.year
df = df.drop(columns=['Date_Of_Birth'], errors='ignore')

print("Paso 1 completado: columnas irrelevantes eliminadas y 'Age' creada")
df.head()

## 1.1 Limpieza de outliers en el TARGET (`Salary`)

Esta es la corrección más importante del notebook. Si el dataset trae registros con errores de
captura en `Salary` (casi $0, o exageradamente altos), la red **aprende de esos errores** como si
fueran patrones reales — y ningún escalamiento de `X` corrige esto, porque el problema está en `y`.

Se aplican dos criterios:
1. **Regla de negocio explícita**: un salario anual por debajo de `MIN_PLAUSIBLE_SALARY` no es un
   sueldo real.
2. **Método IQR**: valores fuera de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` se consideran outliers estadísticos.

Se **eliminan** las filas (no se imputan), porque imputar el target introduce sesgo artificial.

In [ ]:
print("Salary - estadísticos ANTES de limpiar:")
print(df['Salary'].describe())

MIN_PLAUSIBLE_SALARY = 1000  # ajustar según el mercado/moneda real de tu negocio

Q1 = df['Salary'].quantile(0.25)
Q3 = df['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

effective_lower_bound = max(MIN_PLAUSIBLE_SALARY, lower_fence)

n_before = len(df)
mask_valid_salary = (df['Salary'] >= effective_lower_bound) & (df['Salary'] <= upper_fence)
removed = df.loc[~mask_valid_salary, 'Salary']

print(f"\nLímite inferior efectivo: ${effective_lower_bound:,.2f}")
print(f"Límite superior (IQR):    ${upper_fence:,.2f}")
print(f"Filas eliminadas por Salary fuera de rango: {len(removed)} de {n_before} ({len(removed)/n_before*100:.2f}%)")
if len(removed) > 0:
    print("Ejemplos de valores eliminados:", sorted(removed.tolist())[:10])

df = df.loc[mask_valid_salary].reset_index(drop=True)

print("\nSalary - estadísticos DESPUÉS de limpiar:")
print(df['Salary'].describe())

## 2. Imputación inteligente + Tratamiento de outliers en Age

Orden importante:
1. Primero imputamos las **categóricas** (moda).
2. Después tratamos outliers de `Age` y la imputamos (mediana agrupada por `Role`).
3. Por último imputamos `Experience` (mediana agrupada por `Role` + `Qualification`).

In [ ]:
# === 2.1 Imputación de categóricas (moda) ===
for cat_col in ['Qualification', 'Role', 'University']:
    if cat_col in df.columns:
        df[cat_col] = df[cat_col].fillna(df[cat_col].mode()[0])

df['Cert'] = df['Cert'].fillna('No')

# === 2.2 Tratamiento de outliers / valores ilógicos en Age ===
MIN_VALID_AGE = 18
MAX_VALID_AGE = 70

outliers_age = ((df['Age'] < MIN_VALID_AGE) | (df['Age'] > MAX_VALID_AGE)).sum()
print(f"Registros con Age fuera de rango [{MIN_VALID_AGE}, {MAX_VALID_AGE}]: {outliers_age}")

df.loc[(df['Age'] < MIN_VALID_AGE) | (df['Age'] > MAX_VALID_AGE), 'Age'] = np.nan
df['Age'] = df.groupby('Role')['Age'].transform(lambda x: x.fillna(x.median()))
df['Age'] = df['Age'].fillna(df['Age'].median())

# === 2.3 Imputación de Experience (mediana agrupada) ===
df['Experience'] = df.groupby(['Role', 'Qualification'])['Experience'].transform(
    lambda x: x.fillna(x.median())
)
df['Experience'] = df['Experience'].fillna(df['Experience'].median())

inconsistentes = (df['Experience'] > df['Age']).sum()
print(f"Registros con Experience > Age (inconsistentes): {inconsistentes}")
df.loc[df['Experience'] > df['Age'], 'Experience'] = df['Age'] - MIN_VALID_AGE
df['Experience'] = df['Experience'].clip(lower=0)

print("\nNulos restantes por columna:")
print(df.isnull().sum())

## 3. Feature Engineering

Mismas variables que en v14. **Nota de fuga de datos:** no se crea `Salary_Per_Hour` como feature
de entrada porque es una transformación directa del target — incluirla sería fuga de datos
(data leakage).

In [ ]:
QUALIFICATION_MAP = {'Bsc': 1, 'Msc': 2, 'PhD': 3}
UNIVERSITY_MAP = {'Tier3': 1, 'Tier2': 2, 'Tier1': 3}

df['Qualification_Level'] = df['Qualification'].map(QUALIFICATION_MAP)
df['University_Tier'] = df['University'].map(UNIVERSITY_MAP).fillna(2)
df['Is_Senior'] = (df['Role'] == 'Senior').astype(int)
df['Has_Cert'] = (df['Cert'] == 'Yes').astype(int)

df['Experience_Age_Ratio'] = df['Experience'] / (df['Age'] + 1e-6)
df['Exp_Qual_Interaction'] = df['Experience'] * df['Qualification_Level']
df['Age_Qual_Interaction'] = df['Age'] * df['Qualification_Level']
df['Experience_Log'] = np.log1p(df['Experience'])

AGE_BINS = [0, 25, 35, 45, 55, 120]
EXP_BINS = [-1, 2, 5, 10, 20, 100]

df['Age_Bin'] = pd.cut(df['Age'], bins=AGE_BINS, labels=[1, 2, 3, 4, 5]).astype(int)
df['Experience_Bin'] = pd.cut(df['Experience'], bins=EXP_BINS, labels=[1, 2, 3, 4, 5]).astype(int)

df['Seniority_Score'] = (
    df['Qualification_Level'] * 2
    + df['Is_Senior'] * 3
    + df['Has_Cert'] * 1
    + df['Experience_Log']
)

df = df.drop(columns=['Qualification', 'University', 'Role', 'Cert'], errors='ignore')

print("Paso 3 completado: Feature Engineering")
print("Columnas finales:", df.columns.tolist())

## 4. Preparación de datos: target, split estratificado y escalamiento de X **y** de y

**Diferencia clave respecto a `Model_7`:** aquí se escala también el target (`y`), con su propio
`StandardScaler`. Esto le da a la red un problema numéricamente mucho más fácil de aprender (salidas
centradas en 0, desviación estándar 1). Las predicciones se revierten con `inverse_transform` **antes**
de calcular cualquier métrica o de guardar/mostrar resultados, así nunca se reporta ni se guarda un
número en la escala equivocada.

In [ ]:
# Target
y = df['Salary'].values.reshape(-1, 1)

# X: solo columnas numéricas, sin Salary
X = df.select_dtypes(include=[np.number]).drop(columns=['Salary'], errors='ignore').copy()
feature_names = X.columns.tolist()

print("Features usadas por el modelo:")
print(feature_names)

assert "Experience" in feature_names, (
    "'Experience' no está en las columnas del modelo: revisa por qué se perdió en el pipeline."
)

# Split estratificado por cuantiles de salario
salary_bins = pd.qcut(y.flatten(), q=10, labels=False, duplicates='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=salary_bins
)

# Escalamiento de X
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Escalamiento de y (la corrección clave frente a Model_7)
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print(f"\nTrain: {len(y_train)} | Test: {len(y_test)}")
print(f"Salary train -> media: {y_train.mean():,.0f}, std: {y_train.std():,.0f}")
print(f"Salary train escalado -> media: {y_train_scaled.mean():.4f}, std: {y_train_scaled.std():.4f}")

## 5. Modelado: Red Neuronal (Keras)

Arquitectura similar a `Model_7` (2 capas ocultas de 64 neuronas, ReLU, optimizador Adam), pero con
las correcciones clave:
- Entrena sobre `y` **escalado**, no en dólares crudos.
- `EarlyStopping` monitorea `val_loss` (métrica de **validación**, calculada sobre un `validation_split`
  separado del training), no el `loss` de entrenamiento. Esto detecta sobreajuste real, no solo que el
  training loss dejó de bajar.
- Se agrega `Dropout` como regularización adicional (ayuda a que la red no memorice).

In [ ]:
def build_model(input_dim):
    model = Sequential([
        Dense(64, activation='relu', input_shape=[input_dim]),
        Dropout(0.1),
        Dense(64, activation='relu'),
        Dense(1)  # salida lineal: predice el salario ESCALADO
    ])
    model.compile(loss='mse', optimizer='adam', metrics=['mae'])
    return model

model = build_model(X_train_scaled.shape[1])
model.summary()

early_stop = EarlyStopping(
    monitor='val_loss',     # <-- corrección clave: valida, no solo mira el training loss
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train_scaled,
    validation_split=0.15,  # <-- separa un 15% del train como validación real
    epochs=500,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

print(f"\nEntrenamiento terminado. Épocas reales usadas: {len(history.history['loss'])}")
print(f"Mejor val_loss alcanzado: {min(history.history['val_loss']):.4f}")

## 6. Evaluación: predicciones revertidas a dólares reales

In [ ]:
def calc_metrics(y_true, y_pred):
    return {
        "MAPE": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

# Predicciones en escala normalizada...
y_pred_train_scaled = model.predict(X_train_scaled, verbose=0)
y_pred_test_scaled = model.predict(X_test_scaled, verbose=0)

# ...y reversión INMEDIATA a dólares reales antes de medir nada
y_pred_train = scaler_y.inverse_transform(y_pred_train_scaled).flatten()
y_pred_test = scaler_y.inverse_transform(y_pred_test_scaled).flatten()
y_train_real = y_train.flatten()
y_test_real = y_test.flatten()

final_train = calc_metrics(y_train_real, y_pred_train)
final_test = calc_metrics(y_test_real, y_pred_test)

print("="*75)
print("RESULTADOS FINALES - Red Neuronal (v15 NN)")
print("="*75)
print(f"{'Métrica':<10}{'Train':>15}{'Test':>15}")
print(f"{'MAPE':<10}{final_train['MAPE']:>14.2f}%{final_test['MAPE']:>14.2f}%")
print(f"{'MAE':<10}${final_train['MAE']:>13,.0f}{'':>1}${final_test['MAE']:>13,.0f}")
print(f"{'RMSE':<10}${final_train['RMSE']:>13,.0f}{'':>1}${final_test['RMSE']:>13,.0f}")
print(f"{'R2':<10}{final_train['R2']:>15.4f}{final_test['R2']:>15.4f}")
print("="*75)
print("Si Train es mucho mejor que Test, hay señal de overfitting.")

print("\nEjemplo de predicciones (muestra de 10, ya en dólares reales):")
sample_idx = np.random.choice(len(y_test_real), size=min(10, len(y_test_real)), replace=False)
comparison_sample = pd.DataFrame({
    'Real': y_test_real[sample_idx].round(0),
    'Predicho': y_pred_test[sample_idx].round(0)
})
print(comparison_sample.to_string(index=False))

## 7. Curva de pérdida (train vs. validación)

Graficar `loss` (train) contra `val_loss` (validación) es la forma correcta de ver si hay
sobreajuste: si `val_loss` deja de bajar (o sube) mientras `loss` sigue bajando, la red está
memorizando en vez de generalizar — justo lo que `EarlyStopping` con `monitor='val_loss'` está
diseñado para detectar y detener a tiempo.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(history.history['loss'], label='Train loss (MSE escalado)')
plt.plot(history.history['val_loss'], label='Val loss (MSE escalado)')
plt.title('Curva de pérdida - Red Neuronal (v15 NN)')
plt.xlabel('Épocas')
plt.ylabel('Loss (MSE, escala normalizada)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Importancia de variables (Permutation Importance)

Los pesos de una red neuronal no se interpretan directamente como la importancia de variables de
un árbol. En vez de eso, se usa **permutation importance**: se mezcla (baraja) una columna a la vez
en el set de test y se mide cuánto empeora el error del modelo. Si al mezclar una columna el error
sube mucho, esa columna era importante; si casi no cambia, el modelo no dependía tanto de ella.

In [ ]:
class KerasRegressorWrapper(BaseEstimator, RegressorMixin):
    """Envoltorio mínimo para que permutation_importance de sklearn pueda llamar
    a un modelo de Keras como si fuera un estimador de sklearn."""
    def __init__(self, keras_model, scaler_y):
        self.keras_model = keras_model
        self.scaler_y = scaler_y

    def fit(self, X, y):
        return self  # ya está entrenado, no se reentrena aquí

    def predict(self, X):
        pred_scaled = self.keras_model.predict(X, verbose=0)
        return self.scaler_y.inverse_transform(pred_scaled).flatten()

wrapped_model = KerasRegressorWrapper(model, scaler_y)

perm_result = permutation_importance(
    wrapped_model, X_test_scaled, y_test_real,
    n_repeats=10, random_state=RANDOM_STATE, scoring='neg_mean_absolute_error'
)

imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": perm_result.importances_mean
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(9, 7))
plt.barh(imp_df["Feature"], imp_df["Importance"], color="#2c7fb8")
plt.title("Importancia de variables (Permutation Importance) - Red Neuronal")
plt.xlabel("Incremento del MAE al mezclar la variable (mayor = más importante)")
plt.tight_layout()
plt.show()

imp_df.sort_values("Importance", ascending=False)

## 9. Guardado del modelo (listo para app)

In [ ]:
MODEL_PATH = "salary_nn_model_v15.keras"
model.save(MODEL_PATH)
print(f"Modelo guardado: {MODEL_PATH}")

metadata = {
    "model_path": MODEL_PATH,
    "scaler_X": scaler_X,
    "scaler_y": scaler_y,
    "feature_names": feature_names,
    "qualification_map": QUALIFICATION_MAP,
    "university_map": UNIVERSITY_MAP,
    "age_bins": AGE_BINS,
    "exp_bins": EXP_BINS,
    "age_bounds": {"min": MIN_VALID_AGE, "max": MAX_VALID_AGE},
    "metrics": {"train": final_train, "test": final_test},
    "feature_importances": dict(zip(feature_names, [float(x) for x in perm_result.importances_mean])),
    "salary_range": {
        "min_observed": float(y_train_real.min()),
        "max_observed": float(y_train_real.max()),
        "p01": float(np.percentile(y_train_real, 1)),
        "p99": float(np.percentile(y_train_real, 99)),
    },
}

with open("salary_nn_metadata_v15.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Metadata guardada: salary_nn_metadata_v15.pkl")
print(f"MAE test: \${final_test['MAE']:,.0f} | MAPE test: {final_test['MAPE']:.2f}% | R2 test: {final_test['R2']:.4f}")

## 10. Notas finales

- **Escalamiento de `y`**: la corrección más importante frente a `Model_7`. Al escalar el target con
  `StandardScaler` y revertir con `inverse_transform` antes de medir/mostrar cualquier resultado, la
  red aprende un problema numéricamente estable y las predicciones finales están garantizadas en la
  escala real de dólares.
- **`EarlyStopping(monitor='val_loss')` + `validation_split`**: en `Model_7` el `EarlyStopping`
  vigilaba el propio `loss` de entrenamiento, lo cual permite parar en un mínimo que no generaliza.
  Aquí se separa explícitamente un conjunto de validación y se vigila el error ahí, que es la forma
  correcta de detectar sobreajuste.
- **Limpieza del target (`Salary`)**: igual que en v14, se eliminan registros con salarios irreales
  antes de entrenar — esto es lo que evita que la red aprenda patrones de datos corruptos.
- **`Dropout(0.1)`**: regularización adicional para reducir el riesgo de que la red memorice el
  training set.
- **Permutation Importance** en vez de pesos crudos: es la forma correcta de estimar importancia de
  variables en una red neuronal, ya que los pesos por sí solos no son directamente interpretables
  como en un árbol.
- **`Salary_Per_Hour`** deliberadamente no se usa como feature (fuga de datos), igual que en v14.
- Para reentrenar con datos nuevos, basta con reemplazar la URL de carga y correr el notebook
  completo de nuevo.